# B0-SCALAR: REINVENT4 RL без Pareto — 3 seeds

Это **обновлённый baseline B0**, построенный на той же генеративной архитектуре, что и M1, но **без Pareto-оптимизации и без priority-профилей**.

Цель — сделать максимально контролируемое сравнение `B0 vs M1`: оставить одинаковыми генератор, prior, 7 surrogate evaluators, uncertainty, AD, SAS и independent oracle, а изменить только способ объединения 7 целей.

### M1
`7 priority agents → Pareto rank → priority tie-break → reward`

### B0-SCALAR
`1 agent → 7 conservative utilities → equal-weight scalar reward`

$$
R_{B0}(x)=
\begin{cases}
\left(\prod_{k=1}^{7} u_k(x)\right)^{1/7}, & x\in AD_A\cap AD_B,\ SAS\le5\\
0, & \text{иначе}
\end{cases}
$$

где $u_k$ — utility свойства с учётом uncertainty (`beta = 0.5`).

**Важно:** independent oracle не участвует в reward и применяется только после финального sampling.


## Почему это хороший B0 для сравнения с M1

Старый B0 на crossover/mutation сильно отличался от M1 сразу по нескольким компонентам. Этот B0-SCALAR — более строгий baseline/абляция:

- тот же REINVENT4 prior;
- та же DAP learning strategy;
- те же 7 evaluator-моделей;
- те же thresholds;
- те же uncertainty margins;
- тот же AD gate;
- тот же `SAScore <= 5`;
- те же 7 independent oracle-моделей;
- те же 3 random seed;
- **нет Pareto rank**;
- **нет priority agents**;
- все 7 objectives объединяются симметрично в одну scalar reward.

Поэтому разница `M1 − B0-SCALAR` гораздо лучше характеризует вклад Pareto/multi-priority механизма.


## 1. Параметры эксперимента и одинаковый budget с M1

M1 использует `7 profiles × 20 steps × 64 = 8960` RL-scored molecule rows на seed.

Чтобы B0 не получил меньший budget, здесь используется один scalar agent:

`140 steps × 64 = 8960` molecule rows на seed.

На 3 seeds оба метода имеют:

`3 × 8960 = 26 880` training molecule rows.

Финальный sampling тоже выровнен:

- M1: `7 × 500 = 3500` rows на seed;
- B0-SCALAR: `3500` unique sampled molecules на seed.


In [ ]:
# ===================== НАСТРОЙКИ =====================
SEEDS = [42, 101, 2024]
STEPS_PER_SEED = 140
BATCH_SIZE = 64
SAMPLE_PER_SEED = 3500

RUN_SETUP = True
RUN_CHECK = True
RUN_SMOKE = True
RUN_MAIN_B0 = True

import shutil, subprocess
HAS_NVIDIA = shutil.which("nvidia-smi") is not None
if HAS_NVIDIA:
    gpu_info = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True
    ).stdout.strip()
    print("GPU:", gpu_info)
else:
    print("GPU не обнаружен")

if RUN_MAIN_B0 and not HAS_NVIDIA:
    raise RuntimeError(
        "Для полного B0-SCALAR эксперимента включите GPU в Colab: "
        "Runtime -> Change runtime type -> T4/L4 GPU, затем запустите notebook сверху."
    )

PROCESSOR = "cu126" if HAS_NVIDIA else "cpu"
DEVICE = "cuda:0" if HAS_NVIDIA else "cpu"

planned_train = len(SEEDS) * STEPS_PER_SEED * BATCH_SIZE
planned_final = len(SEEDS) * SAMPLE_PER_SEED
print("Seeds:", SEEDS)
print("Processor:", PROCESSOR, "Device:", DEVICE)
print("Training molecule-rows:", f"{planned_train:,}")
print("Final sampled rows:", f"{planned_final:,}")
assert planned_train == 26880, "Default budget should match M1: 26,880"


## 2. Разворачиваем воспроизводимую сборку

Код B0-SCALAR встроен в notebook. Тяжёлые `.joblib` evaluator/oracle модели скачиваются из pinned Case commit во время `setup`.


In [ ]:
import base64, zipfile, pathlib, shutil
ROOT = pathlib.Path("/content/REINVENT4_MOST_B0_SCALAR_RL_3SEEDS_V1")
ARCHIVE = pathlib.Path("/content/REINVENT4_MOST_B0_SCALAR_RL_3SEEDS_V1.zip")

_payload = "".join([
'...',
])
ARCHIVE.write_bytes(base64.b64decode(_payload))
if ROOT.exists():
    shutil.rmtree(ROOT)
with zipfile.ZipFile(ARCHIVE) as z:
    z.extractall("/content")
print("Bundle extracted:", ROOT)
print((ROOT / "README_RU.md").read_text(encoding="utf-8")[:2500])


## 3. Проверка: в reward действительно нет Pareto

В этой версии нет rank-based reward, priority profiles или tie-breaker. Scalar reward — геометрическое среднее семи utilities.


In [ ]:
from pathlib import Path
core = (ROOT / "scoring" / "scalar_core.py").read_text(encoding="utf-8")
run_src = (ROOT / "run.py").read_text(encoding="utf-8")
config_src = (ROOT / "config_builder.py").read_text(encoding="utf-8")

print("scalar_core.py exists:", (ROOT / "scoring" / "scalar_core.py").exists())
print("Pareto client exists:", (ROOT / "scoring" / "pareto_client.py").exists())
print("Pareto daemon exists:", (ROOT / "scoring" / "pareto_daemon.py").exists())
print("Equal-weight reward present:", "equal_weight_geometric_scalarization" in core)
print("Priority profiles used:", "priority_profiles_used': True" in run_src)

assert not (ROOT / "scoring" / "pareto_client.py").exists()
assert not (ROOT / "scoring" / "pareto_daemon.py").exists()
assert "equal_weight_geometric_scalarization" in core
print("OK: B0-SCALAR использует scalar reward без Pareto.")


## 4. Надёжный запуск тяжёлых команд в Colab

Полный stdout пишется в лог, чтобы не перегружать Colab websocket. В интерфейс выводится heartbeat и хвост лога.


In [ ]:
import subprocess, pathlib, shutil, os, sys, time

LOG_DIR = ROOT / "colab_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

def _tail_text(path, n=160):
    p = pathlib.Path(path)
    if not p.exists():
        return f"<log not found: {p}>"
    lines = p.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-n:])

def _resource_status():
    parts=[]
    try:
        info={}
        for line in pathlib.Path('/proc/meminfo').read_text().splitlines():
            k,v=line.split(':',1); info[k]=v.strip()
        avail=int(info['MemAvailable'].split()[0])/1024/1024
        total=int(info['MemTotal'].split()[0])/1024/1024
        parts.append(f"RAM {avail:.1f}/{total:.1f} GiB available")
    except Exception: pass
    try:
        du=shutil.disk_usage('/content'); parts.append(f"disk {du.free/1024**3:.1f} GiB free")
    except Exception: pass
    if shutil.which('nvidia-smi'):
        try:
            x=subprocess.run(['nvidia-smi','--query-gpu=memory.used,memory.total','--format=csv,noheader,nounits'],capture_output=True,text=True,timeout=5)
            if x.returncode==0: parts.append('GPU MiB '+x.stdout.strip().replace('\n','; '))
        except Exception: pass
    return ' | '.join(parts)

def run_logged(cmd, log_name, cwd=ROOT, check=True, heartbeat_seconds=30):
    log_path=LOG_DIR/log_name
    cmd=[str(x) for x in cmd]
    print('+',' '.join(cmd)); print('Full log:',log_path)
    env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env.setdefault('PYTHONWARNINGS','ignore::DeprecationWarning')
    started=time.time()
    with log_path.open('w',encoding='utf-8',buffering=1) as log:
        p=subprocess.Popen(cmd,cwd=str(cwd),stdout=log,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',env=env)
        last=-heartbeat_seconds
        while True:
            rc=p.poll(); elapsed=time.time()-started
            if elapsed-last>=heartbeat_seconds:
                print(f"[{elapsed/60:.1f} min] running | {_resource_status()}",flush=True); last=elapsed
            if rc is not None: break
            time.sleep(2)
    if rc!=0:
        print(f"FAILED, return code={rc}")
        print("\n--- REAL ERROR: LAST LOG LINES ---")
        print(_tail_text(log_path,180))
        if check: raise RuntimeError(f"Command failed. Full log: {log_path}")
        return rc
    print(f"DONE in {(time.time()-started)/60:.1f} min")
    print(_tail_text(log_path,25))
    return rc

def ensure_uv_python312():
    uv=shutil.which('uv')
    if uv is None:
        subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
        uv=shutil.which('uv')
    subprocess.run([uv,'python','install','3.12'],check=True)
    r=subprocess.run([uv,'python','find','3.12'],capture_output=True,text=True,check=True)
    py=r.stdout.strip().splitlines()[-1].strip()
    print('Pinned setup Python:',py)
    return py

RUNNER_PY=ensure_uv_python312()


## 5. Setup: REINVENT4 + 7 evaluators + 7 oracle

`setup` создаёт отдельные environments, скачивает pinned REINVENT4 prior и Case models, строит AD cache и выполняет unit tests/preflight.


In [ ]:
def setup_with_one_clean_retry():
    cmd=[RUNNER_PY,'run.py','setup','--processor',PROCESSOR]
    def clean_partial():
        for name in ['.venv-engine','.venv-evaluators']:
            p=ROOT/name
            if p.exists(): shutil.rmtree(p,ignore_errors=True)
        for p in ROOT.rglob('*.partial'): p.unlink(missing_ok=True)
    try:
        run_logged(cmd,'setup_attempt1.log')
    except Exception as first:
        print('First setup attempt failed; cleaning reproducible partial state and retrying once.')
        clean_partial()
        try:
            run_logged(cmd,'setup_attempt2.log')
        except Exception:
            print('--- attempt 1 tail ---'); print(_tail_text(LOG_DIR/'setup_attempt1.log',120))
            print('--- attempt 2 tail ---'); print(_tail_text(LOG_DIR/'setup_attempt2.log',180))
            raise RuntimeError('Setup failed twice; use REAL ERROR log lines above.') from first

if RUN_SETUP:
    setup_with_one_clean_retry()
else:
    print('RUN_SETUP=False')


## 6. Unit tests, check и smoke

Проверяется, что:

- Case commit зафиксирован;
- ровно 7 evaluator + 7 oracle;
- oracle не входит в reward;
- reward симметричен относительно семи objectives;
- rank-based evaluator files отсутствуют;
- REINVENT способен сделать короткий RL update и сохранить checkpoint.


In [ ]:
if RUN_CHECK:
    run_logged([RUNNER_PY,'run.py','test'],'unit_tests.log')
    run_logged([RUNNER_PY,'run.py','check','--seed',str(SEEDS[0])],'check.log')
if RUN_SMOKE:
    run_logged([
        RUNNER_PY,'run.py','smoke','--steps','2','--batch-size','16',
        '--device',DEVICE,'--seed',str(SEEDS[0])
    ],'smoke.log')


## 7. Основной B0-SCALAR эксперимент

Для каждого seed обучается **один** REINVENT agent на 140 шагов. На каждом RL batch:

1. REINVENT генерирует SMILES;
2. 7 surrogate evaluator-ов считают predictions + uncertainties;
3. predictions переводятся в 7 conservative utilities;
4. если кандидат вне обоих AD или `SAS > 5`, reward = 0;
5. иначе reward = equal-weight geometric mean 7 utilities;
6. REINVENT обновляет policy;
7. после обучения семплируются 3500 unique molecules;
8. только после этого запускаются 7 independent oracle-моделей.


In [ ]:
if RUN_MAIN_B0:
    cmd=[
        RUNNER_PY,'run.py','experiment',
        '--steps',str(STEPS_PER_SEED),
        '--batch-size',str(BATCH_SIZE),
        '--n',str(SAMPLE_PER_SEED),
        '--device',DEVICE,
        '--seeds',*map(str,SEEDS),
    ]
    run_logged(cmd,'experiment_b0_scalar_3seeds.log')
else:
    print('RUN_MAIN_B0=False')


## 8. Загружаем результаты

Главные показатели для последующего B0↔M1 сравнения:

- `JSR_Oracle` — независимый joint success;
- `JSR_Oracle_reliable_AD_SAS` — joint success после AD+SAS;
- `JSR_Surrogate_raw`;
- `Robust_Surrogate_Rate`;
- `Novelty`;
- `Internal_Diversity`;
- `AD_Both_Rate`;
- `SAScore_Pass_Rate`;
- evaluator↔oracle agreement.


In [ ]:
import json, pandas as pd, numpy as np
from pathlib import Path

latest=ROOT/'runs'/'latest_b0_scalar_experiment.json'
if not latest.exists():
    raise FileNotFoundError('Нет B0-SCALAR результатов. Сначала выполните основной experiment.')
EXP=Path(json.loads(latest.read_text(encoding='utf-8'))['experiment'])
AGG=EXP/'aggregate'
print('Experiment:',EXP)
seed_metrics=pd.read_csv(AGG/'seed_metrics.csv')
mean_std=pd.read_csv(AGG/'seed_metrics_mean_std.csv')
summary=json.loads((AGG/'summary.json').read_text(encoding='utf-8'))
display(seed_metrics)
display(mean_std.T.rename(columns={0:'value'}))
print(json.dumps({k:v for k,v in summary.items() if k not in {'per_seed','mean_std'}},indent=2,ensure_ascii=False))


## 9. Mean ± std по 3 seed

Для защиты основными будут независимые Oracle-метрики. Scalar reward/reward curve — доказательство того, что agent адаптировался к evaluator objective, но не является независимым подтверждением качества.


In [ ]:
metrics=[
    'JSR_Oracle','JSR_Oracle_reliable_AD_SAS','JSR_Surrogate_raw',
    'Robust_Surrogate_Rate','Novelty','Internal_Diversity',
    'AD_Both_Rate','SAScore_Pass_Rate','Evaluator_Oracle_Joint_Agreement',
    'Mean_Scalar_Reward'
]
rows=[]
for m in metrics:
    vals=pd.to_numeric(seed_metrics[m],errors='coerce').dropna().to_numpy(float)
    rows.append({'metric':m,'mean':vals.mean(),'std':vals.std(ddof=0)})
report=pd.DataFrame(rows)
display(report)


## 10. Графики B0-SCALAR по seed


In [ ]:
import matplotlib.pyplot as plt
plot_cols=['JSR_Oracle','JSR_Oracle_reliable_AD_SAS','AD_Both_Rate','Novelty','Internal_Diversity']
for col in plot_cols:
    fig,ax=plt.subplots(figsize=(7,4))
    ax.bar(seed_metrics['seed'].astype(str),seed_metrics[col])
    ax.set_title(f'B0-SCALAR: {col} by seed')
    ax.set_xlabel('seed'); ax.set_ylabel(col)
    ax.set_ylim(0,1 if seed_metrics[col].max()<=1 else None)
    plt.show()


## 11. Что означает результат B0-SCALAR

Этот эксперимент сам по себе не должен называться «M1 без Pareto победил/проиграл». Его задача — создать **контролируемую точку сравнения**.

После получения M1 и B0-SCALAR сравниваем:

$$\Delta JSR_{Oracle}=JSR_{Oracle}^{M1}-JSR_{Oracle}^{B0-scalar}$$

и одновременно:

- `Δ Reliable Oracle JSR`;
- `Δ Diversity`;
- `Δ Novelty`;
- `Δ AD_Both`.

Если M1 выше по Oracle JSR, но ниже по reliable Oracle JSR из-за AD, это означает, что rank-based multi-objective search увеличивает формальный success, но сильнее уходит в extrapolation.

Если M1 выше и по `JSR_Oracle_reliable_AD_SAS`, при сопоставимой diversity, это уже более сильное доказательство вклада M1.


## 12. Экспорт результатов


In [ ]:
from pathlib import Path
result_zip=Path('/content/REINVENT4_MOST_B0_SCALAR_RL_3SEEDS_RESULTS.zip')
ckpt_zip=Path('/content/REINVENT4_MOST_B0_SCALAR_RL_3SEEDS_CHECKPOINTS.zip')
print('Results ZIP:',result_zip, result_zip.exists())
print('Checkpoints ZIP:',ckpt_zip, ckpt_zip.exists())

# Для Colab: раскомментируйте при необходимости.
# from google.colab import files
# files.download(str(result_zip))
